In [1]:
import h5py
import numpy as np
import os
import pandas as pd
import pathlib
import pickle
import scipy.stats as stats
import soxr
#! change below to spatial_attn_lighting if want to use with modular 
import src.spatial_attn_lightning as attn_tracking_lightning
import src.audio_transforms as at
import torch
import yaml

import argparse
from argparse import ArgumentParser
from corpus.speaker_room_dataset import SpeakerRoomDataset
from tqdm.auto import tqdm
from datetime import datetime
import sys 

import IPython.display as ipd
%matplotlib inline
import matplotlib.pyplot as plt


os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

In [11]:
## Check df for vaidation rooms:
df_room = pd.read_pickle('/om2/user/msaddler/spatial_audio_pipeline/assets/brir/v00/manifest_room.pdpkl')
df_brir = pd.read_pickle('/om2/user/msaddler/spatial_audio_pipeline/assets/brir/v00/manifest_brir.pdpkl')
df_brir = df_brir[df_brir.src_elev >= 0]
# use val split subset 
df_brir = df_brir[df_brir.index_room >= 1800]
df_room = df_room[df_room.index_room >= 1800]
print(f"{len(df_brir)} brir files in val split")
print(f"{len(df_room)} room files in val split")

# check for valid rooms with src distance of 1.4m
df_brir = df_brir[df_brir.src_dist == 1.4]
df_room = df_room[df_room.index_room.isin(df_brir.index_room)]
print(f"{len(df_brir)} brir files in val split with src dist of 1.4m")
print(f"{len(df_room)} room files in val split with src dist of 1.4m")

201600 brir files in val split
200 room files in val split
100800 brir files in val split with src dist of 1.4m
200 room files in val split with src dist of 1.4m


In [7]:
df_brir.columns

Index(['buffer', 'c', 'dur', 'head_azim', 'head_pos_xyz',
       'incorporate_lead_zeros', 'index_brir', 'index_room', 'room_dim_xyz',
       'room_materials', 'sr', 'src_azim', 'src_dist', 'src_elev',
       'use_highpass', 'use_hrtf_symmetry', 'use_jitter', 'use_log_distance'],
      dtype='object')

## Dev front-back test placement 

In [2]:
class Spatialize(torch.nn.Module):
    def __init__(self, ir, model_sr=50_000):
        super(Spatialize, self).__init__()
        ir = torch.flip(torch.from_numpy(ir), dims=[0]).float()
        self.n_taps = ir.shape[0]
        ir = ir.T.unsqueeze(1)
        # set center crop of 2.5 seconds relative to model_sr
        self.start_frame = int(model_sr * 0.25)
        self.end_frame = int(model_sr * 2.75)

        self.register_buffer("ir", ir)

    def forward(self, words):
        n_words = words.shape[0]
        # pad last dim of words with ir.shape[0] - 1 zeros
        words_padded = torch.nn.functional.pad(words, (self.n_taps - 1, 0))
        spatialized = torch.nn.functional.conv1d(words_padded.view(n_words, 1, -1), self.ir)
        # resize to desired shape
        spatialized = spatialized[:, :, self.start_frame:self.end_frame]
        return spatialized

In [3]:
### Get most recent config
config_path = "config/binaural_attn/word_task_half_co_loc_v07.yaml"
ckpt_path = "attn_cue_models/word_task_half_co_loc_v07/checkpoints/epoch=2-step=46074.ckpt"

In [4]:

config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)
config['num_workers'] = 2
config['hparas']['batch_size'] = 2 # config['data']['loader']['batch_size'] // args.gpus
config['noise_kwargs']['low_snr'] = 0
config['noise_kwargs']['high_snr'] = 0
# get model input sr for brir resampling
model_in_sr = config['audio']['rep_kwargs']['sr']

In [5]:
model = attn_tracking_lightning.BinauralAttentionModule.load_from_checkpoint(checkpoint_path=ckpt_path, config=config, strict=False).cuda()


num_classes={'num_words': 800}


/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/torchaudio/functional/functional.py:1371: UserWarning: "kaiser_window" resampling method name is being deprecated and replaced by "sinc_interp_kaiser" in the next release. The default behavior remains unchanged.
  warnings.warn(


center_crop=True
binaural=True
Binaural cochleagram
using FIR cochleagram


In [6]:
audio_transforms = at.AudioCompose([
                    at.AudioToTensor(),
                    at.BinauralCombineWithRandomDBSNR(low_snr=0,    # is 0 dB
                                                      high_snr=0), # is 0 dB 
                    at.BinauralRMSNormalizeForegroundAndBackground(rms_level=0.02), # 20 * np.log10(0.02/20e-6) = 60 dB SPL 
            ])


audio_transforms = audio_transforms.cuda()
model = model.eval()
coch_gram = model.coch_gram.cuda()

In [7]:
dataset = SpeakerRoomDataset(manifest_path='/om2/user/rphess/Auditory-Attention/final_binaural_manifest.pkl',
                            excerpt_path='/om2/user/msaddler/spatial_audio_pipeline/assets/swc/manifest_all_words.pdpkl',
                            cue_type='voice_and_location',
                            sr=model_in_sr) 
# dataloader = torch.utils.data.DataLoader(dataset, batch_size=config['hparas']['batch_size'], shuffle=False, num_workers=config['num_workers'])
dataloader = torch.utils.data.DataLoader(dataset,
                                        batch_size=24,
                                        shuffle=False,
                                        num_workers=2)


In [8]:
## Use room 1 for rotated head position - maps to what we can do with humans 
room_ix = 1

new_room_manifest = pd.read_pickle('/om2/user/msaddler/spatial_audio_pipeline/assets/brir/mit_bldg46room1004/manifest_brir.pdpkl')
only14_manifest = new_room_manifest[(new_room_manifest['src_dist'] == 1.4) & (new_room_manifest['index_room'] == room_ix)]

ir_dict = dict()
target_loc = (0, 0)
distractor_loc = [180, 0]
for loc in ['target', 'distractor']:
    if loc == 'target':
        coords = target_loc
    elif loc == 'distractor':
        coords = distractor_loc.copy()
    print(loc)
    print(coords)

    df_row = only14_manifest[(only14_manifest['src_azim'] == coords[0]) & (only14_manifest['src_elev'] == coords[1])]
    h5_fn = f'/om2/user/msaddler/spatial_audio_pipeline/assets/brir/mit_bldg46room1004/room000{room_ix}.hdf5'
    index_brir = df_row['index_brir'].values[0]
    sr_src = df_row['sr'].values[0]
    with h5py.File(h5_fn, 'r') as f:
        brir = f['brir'][index_brir]
    if model_in_sr != sr_src:
        brir = soxr.resample(brir.astype(np.float32), sr_src, model_in_sr)
    ir_dict[loc] = brir.astype(np.float32)


target
(0, 0)
distractor
[180, 0]


In [9]:
tar_brir = Spatialize(ir_dict['target'], model_sr=model_in_sr).cuda()
dist_brir = Spatialize(ir_dict['distractor'], model_sr=model_in_sr).cuda()

# run eval 
accs = []
confs = []

accuracies = []
confusions = []
pred_list = []
true_word_int = []

with torch.no_grad(): 
    for batch in tqdm(dataloader):
        cue, fg, bg, label, confusion = batch

        cue = tar_brir(cue.cuda())
        foreground = tar_brir(fg.cuda())
        background = dist_brir(bg.cuda())

        cue = audio_transforms(cue, None)[0]
        mixture = audio_transforms(foreground, background)[0]

        cue, mixture = coch_gram(cue, mixture)
        logits = model(cue, mixture, None)

        preds = logits.softmax(dim=-1).argmax(dim=-1).cpu().detach().numpy().astype('int')
        true_word = label.numpy().astype('int')
        con_word = confusion.numpy().astype('int')
        accuracy = (preds == true_word).astype('int')
        cons = (preds == con_word).astype('int')
        accuracies.append(accuracy)
        confusions.append(cons)
        pred_list.append(preds)
        true_word_int.append(true_word)
        
accuracies = np.concatenate(accuracies)
confusions = np.concatenate(confusions)
preds = np.concatenate(pred_list)
true_word_int = np.concatenate(true_word_int)


print(f"Accuracy: {accuracies.mean():.3f}")
print(f"Confusions: {confusions.mean():.3f}")


 99%|█████████▉| 131/132 [10:45<00:04,  4.68s/it]/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/torch/overrides.py:110: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_built()'
  torch.has_cuda,
/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/torch/overrides.py:111: UserWarning: 'has_cudnn' is deprecated, please use 'torch.backends.cudnn.is_available()'
  torch.has_cudnn,
/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/torch/overrides.py:117: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  torch.has_mps,
/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/torch/overrides.py:118: UserWarning: 'has_mkldnn' is deprecated, please use 'torch.backends.mkldnn.is_available()'
  torch.has_mkldnn,
100%|██████████| 132/132 [10:59<00:00,  5.00s/it]

Accuracy: 0.467
Confusions: 0.107
